In [1]:
import pandas as pd
import numpy as np
from ucimlrepo import fetch_ucirepo 
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.cluster import KMeans
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split

In [10]:
census_income = fetch_ucirepo(id=20) 
X = census_income.data.features
y = census_income.data.targets.values.ravel()

print("Features:", X.shape)
print("Targets:", y.shape)

Features: (48842, 14)
Targets: (48842,)


In [3]:
X = X.fillna("Unknown")
encoders = {}
for col in X.columns:
    if X[col].dtype == "object":
        le = LabelEncoder()
        X[col] = le.fit_transform(X[col])
        encoders[col] = le

# Escalado
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [11]:
print(X_scaled.shape)

(48842, 14)


In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42)

In [12]:
print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

Train shape: (39073, 14)
Test shape: (9769, 14)


In [5]:
k = 50
kmeans = KMeans(n_clusters=k, random_state=42)
X_train_dist = kmeans.fit_transform(X_train)

print("Distancias:", X_train_dist.shape)

Distancias: (39073, 50)


In [6]:
idxs = np.argmin(X_train_dist, axis=0)
X_representatives = X_train[idxs]
y_representatives = y_train[idxs]

print("Representantes:", X_representatives.shape)


Representantes: (50, 14)


In [7]:
# 6. Entrenar modelo con los representantes
log_reg_representatives = LogisticRegression(max_iter=5000, solver="lbfgs")
log_reg_representatives.fit(X_representatives, y_representatives)

print("Score con representantes:", log_reg_representatives.score(X_test, y_test))

Score con representantes: 0.42552973692291945


In [8]:
# 7. Propagación de etiquetas a todo el cluster
y_train_propagated = np.empty(len(X_train), dtype=y_train.dtype)
for i in range(k):
    y_train_propagated[kmeans.labels_ == i] = y_representatives[i]

In [9]:
# 8. Entrenar con propagación (semi-supervisado)
log_reg_propagated = LogisticRegression(max_iter=5000, solver="lbfgs")
log_reg_propagated.fit(X_train, y_train_propagated)

print("Score con etiquetas propagadas:", log_reg_propagated.score(X_test, y_test))

print("\n--- Reporte ---")
print(classification_report(y_test, log_reg_propagated.predict(X_test)))

Score con etiquetas propagadas: 0.42276589210768756

--- Reporte ---
              precision    recall  f1-score   support

       <=50K       0.60      0.63      0.62      4936
      <=50K.       0.33      0.10      0.16      2478
        >50K       0.29      0.28      0.29      1562
       >50K.       0.14      0.38      0.20       793

    accuracy                           0.42      9769
   macro avg       0.34      0.35      0.31      9769
weighted avg       0.44      0.42      0.41      9769

